# Reto 04 — OPT · Búsqueda de hiperparámetros con Optuna
### Despacho de una microred (solar + eólico + red), bi-objetivo

Este notebook busca los mejores hiperparámetros de **NSGA-II** y **SPEA2** para el problema de despacho de la microred, usando **Optuna (TPE)** en lugar de un grid exhaustivo.

**Estrategia**
- Métrica de calidad: **Hypervolume (HV)** — mayor es mejor.
- Robustez: cada configuración se ejecuta `N_RUNS` veces con semillas distintas; se reporta la **mediana** del HV.
- Hiperparámetros buscados (rangos **continuos**): `eta_c` (SBX), `eta_m` (PM), `crossover_prob`.
- **La población NO se busca**: queda fija en `POPULATION_SIZE`.
- Un *study* de Optuna por algoritmo (se conserva la comparación NSGA-II vs SPEA2).

> **Supuesto de modelado**: el individuo es el **calendario completo** de una ventana de `T_HOURS` horas (2·T variables). Cámbialo en la celda de configuración.

## 0. Instalación de dependencias

In [ ]:
!pip install jmetalpy optuna joblib pandas numpy tqdm

## 1. Imports y configuración

In [ ]:
import os, random, time, json, warnings, logging
import numpy as np
import pandas as pd
import optuna

from jmetal.core.problem import FloatProblem
from jmetal.core.solution import FloatSolution
from jmetal.algorithm.multiobjective.nsgaii import NSGAII
from jmetal.algorithm.multiobjective.spea2 import SPEA2
from jmetal.operator.crossover import SBXCrossover
from jmetal.operator.mutation import PolynomialMutation
from jmetal.util.termination_criterion import StoppingByEvaluations
from jmetal.util.solution import get_non_dominated_solutions
from jmetal.core.quality_indicator import HyperVolume

warnings.filterwarnings('ignore')
logging.getLogger('jmetal').setLevel(logging.ERROR)

Montar Google Drive (en Colab).

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('No estamos en Colab o Drive ya montado:', e)

In [ ]:
# ============================================================
#  CONFIGURACIÓN GLOBAL
# ============================================================
BASE_SEED = 42  # Semilla base para reproducibilidad

# --- Rutas (AJUSTA a tu entorno) ------------------------------------------
DATA_DIR   = '/content/drive/MyDrive/reto4-opt/datos'   # carpeta con los 6 CSV
OUTPUT_DIR = '/content/drive/MyDrive/reto4-opt'          # donde se guardan resultados

# --- Horizonte de optimización --------------------------------------------
# El individuo = calendario completo de la ventana.
# Hay 2*T variables: para cada hora t, x1_t (solar) y x2_t (eólico).
# T_HOURS controla el tamaño de la ventana:
#   24 = un día | 168 = una semana | 720 = un mes | 8760 = año completo
T_HOURS    = 168    # ventana representativa (1 semana) -> 336 variables
START_HOUR = 0      # hora de inicio dentro del año (0 .. 8759 - T_HOURS)
PRED_YEAR  = 2020   # año de las predicciones de potencia (2017-2020 disponibles)

# --- Presupuesto evolutivo ------------------------------------------------
# max_evaluations = POPULATION_SIZE * GENERATIONS (igual para NSGA-II y SPEA2)
GENERATIONS     = 250
POPULATION_SIZE = 100   # FIJO: NO se busca en Optuna (decisión del enunciado)
N_RUNS          = 3     # repeticiones por configuración -> se reporta la MEDIANA del HV

# --- Optuna ----------------------------------------------------------------
N_TRIALS   = 40                    # nº de configuraciones que explora Optuna por algoritmo
ALGORITHMS = ['NSGAII', 'SPEA2']

# Espacio de búsqueda CONTINUO (Optuna/TPE afina mejor que un grid discreto)
SEARCH_SPACE = {
    'eta_c'          : (5.0, 30.0),   # índice de distribución SBX  (5=explorador, 30=explotador)
    'eta_m'          : (5.0, 30.0),   # índice de distribución PM
    'crossover_prob' : (0.7, 1.0),    # probabilidad de cruce SBX (literatura: >=0.7)
}

print('Configuración cargada.')
print(f'  Ventana            : {T_HOURS} h  (desde la hora {START_HOUR})  -> {2*T_HOURS} variables')
print(f'  Año predicciones   : {PRED_YEAR}')
print(f'  Población (fija)   : {POPULATION_SIZE}')
print(f'  Generaciones       : {GENERATIONS}')
print(f'  Runs por config    : {N_RUNS}')
print(f'  Trials Optuna/algo : {N_TRIALS}')

## 2. Carga y alineación de datos

In [ ]:
# ============================================================
#  CARGA Y ALINEACIÓN DE DATOS
# ============================================================
# IMPORTANTE: los ficheros NO comparten fechas absolutas
#   - precios (solar/eólico/red): año 2025, horario  (8760 h)
#   - predicciones de potencia  : 2017-2020, horario  (~4 años)
#   - demanda (restaurante)     : sin fecha, 8760 valores horarios
# Por eso la alineación es POSICIONAL (hora-del-año, índice 0..8759).
# Tomamos un año de predicciones (PRED_YEAR) y emparejamos por posición.

# --- Demanda d_t (kW = kWh/h) ---------------------------------------------
dem = pd.read_csv(os.path.join(DATA_DIR, 'RefBldgFullServiceRestaurantNew2004_v1_3_7_1_6A_USA_MN_MINNEAPOLIS.csv'))
d_full = dem.iloc[:, 0].to_numpy(dtype=float)            # [kWh/h]

# --- Precio de la red a3_t (€/MWh -> €/kWh) --------------------------------
pg = pd.read_csv(os.path.join(DATA_DIR, 'precio2025-peninsula.csv'), sep=';')
a_grid_full = pg['value'].to_numpy(dtype=float) / 1000.0  # [€/kWh]

# --- Precio solar a1_t y eólico a2_t (€/MWh -> €/kWh) ----------------------
ps = pd.read_csv(os.path.join(DATA_DIR, 'precio_solar_mwh.csv'),  sep=';')
pe = pd.read_csv(os.path.join(DATA_DIR, 'precio_eolico_mwh.csv'), sep=';')
a_solar_full = ps['precio_eur_mwh'].to_numpy(dtype=float) / 1000.0  # [€/kWh]
a_wind_full  = pe['precio_eur_mwh'].to_numpy(dtype=float) / 1000.0  # [€/kWh]

# --- Capacidad solar P_solar_t y eólica P_wind_t (kW = kWh/h) --------------
sol = pd.read_csv(os.path.join(DATA_DIR, 'Predicciones_Solar.csv'))
win = pd.read_csv(os.path.join(DATA_DIR, 'Predicciones_Eolico.csv'))
sol['Date'] = pd.to_datetime(sol['Date'])
win['Date'] = pd.to_datetime(win['Date'])
Psolar_full = sol.loc[sol['Date'].dt.year == PRED_YEAR, 'SystemProduction_AS'].to_numpy(dtype=float)[:8760]
Pwind_full  = win.loc[win['Date'].dt.year == PRED_YEAR, 'Power_AE'].to_numpy(dtype=float)[:8760]

# --- Recorte a longitud común y a la ventana de optimización ---------------
N = min(len(d_full), len(a_grid_full), len(a_solar_full), len(a_wind_full),
        len(Psolar_full), len(Pwind_full))
sl = slice(START_HOUR, START_HOUR + T_HOURS)
assert START_HOUR + T_HOURS <= N, f'La ventana excede los datos disponibles (N={N}).'

DEMAND  = d_full[:N][sl]        # d_t      [kWh/h]
P_SOLAR = Psolar_full[:N][sl]   # P_solar_t [kWh/h]
P_WIND  = Pwind_full[:N][sl]    # P_wind_t  [kWh/h]
A_SOLAR = a_solar_full[:N][sl]  # a1_t [€/kWh]
A_WIND  = a_wind_full[:N][sl]   # a2_t [€/kWh]
A_GRID  = a_grid_full[:N][sl]   # a3_t [€/kWh]

T = len(DEMAND)
print(f'Datos alineados (posicionalmente). Horas en la ventana: T = {T}')
print(f'  Demanda      [kWh/h]: media {DEMAND.mean():.1f}  | max {DEMAND.max():.1f}')
print(f'  Cap. solar   [kWh/h]: media {P_SOLAR.mean():.1f} | max {P_SOLAR.max():.1f}')
print(f'  Cap. eólica  [kWh/h]: media {P_WIND.mean():.1f}  | max {P_WIND.max():.1f}')
print(f'  Precio solar [€/kWh]: media {A_SOLAR.mean():.4f}')
print(f'  Precio eólico[€/kWh]: media {A_WIND.mean():.4f}')
print(f'  Precio red   [€/kWh]: media {A_GRID.mean():.4f}')
print(f'  Horas solar mas barato que red : {(A_SOLAR < A_GRID).mean()*100:.1f}%')
print(f'  Horas eólico mas barato que red: {(A_WIND  < A_GRID).mean()*100:.1f}%')
print(f'  Horas en que renovables cubren la demanda: {((P_SOLAR+P_WIND) >= DEMAND).mean()*100:.1f}%')

## 3. Definición del problema

In [ ]:
# ============================================================
#  DEFINICIÓN DEL PROBLEMA  (despacho de microred, bi-objetivo)
# ============================================================
# Traducimos las ecuaciones del despacho al lenguaje de jMetalPy.
#
# Variables (2*T):  [x1_0..x1_{T-1} | x2_0..x2_{T-1}]
#   x1_t = energía comprada al solar  en la hora t   (kWh)
#   x2_t = energía comprada al eólico en la hora t   (kWh)
#   g_t  = d_t - x1_t - x2_t  (resto cubierto por la red, >= 0)
#
# Objetivos (ambos MINIMIZAR):
#   f1 = sum_t [ a1_t*x1_t + a2_t*x2_t + a3_t*(d_t - x1_t - x2_t) ]   (coste, €)
#   f2 = sum_t ( d_t - x1_t - x2_t )                                  (energía de red, kWh)
#
# Restricciones:
#   0 <= x1_t <= P_solar_t  ;  0 <= x2_t <= P_wind_t  ;  x1_t + x2_t <= d_t
#
# Estas restricciones se gestionan por REPARACIÓN dentro de evaluate():
#   1) clip de cada flujo a [0, capacidad]  (cotas y no-negatividad)
#   2) si x1_t + x2_t > d_t  ->  reescalado proporcional para que la suma = d_t
# Motivo: dejar la restricción a los objetivos NO funciona. Si x1+x2 > d_t,
# entonces g_t < 0 y BAJARÍA tanto f1 como f2: el infactible "parece" mejor y
# dominaría al factible. La reparación lo evita conservando el reparto aprendido.

class MicrogridDispatch(FloatProblem):
    """Despacho bi-objetivo de una microred (coste vs energía de red)."""

    def __init__(self, demand, p_solar, p_wind, a_solar, a_wind, a_grid):
        super().__init__()
        self.d  = np.asarray(demand,  dtype=float)
        self.ps = np.asarray(p_solar, dtype=float)
        self.pw = np.asarray(p_wind,  dtype=float)
        self.a1 = np.asarray(a_solar, dtype=float)
        self.a2 = np.asarray(a_wind,  dtype=float)
        self.a3 = np.asarray(a_grid,  dtype=float)
        self.T  = len(self.d)

        # Cotas por gen: solar en [0, P_solar_t], eólico en [0, P_wind_t]
        self.lower_bound    = [0.0] * (2 * self.T)
        self.upper_bound    = list(self.ps) + list(self.pw)
        self.obj_directions = [self.MINIMIZE, self.MINIMIZE]
        self.obj_labels     = ['Coste (€)', 'Energia red (kWh)']

    def number_of_variables(self)   -> int: return 2 * self.T
    def number_of_objectives(self)  -> int: return 2
    def number_of_constraints(self) -> int: return 0
    def name(self)                  -> str: return 'Microgrid Dispatch Optimization'

    # ----------------------------------------------------------------
    def _repair(self, variables):
        """Devuelve (x1, x2) factibles a partir del cromosoma."""
        x  = np.asarray(variables, dtype=float)
        x1 = np.clip(x[:self.T],      0.0, self.ps)   # 0 <= x1 <= P_solar
        x2 = np.clip(x[self.T:],      0.0, self.pw)   # 0 <= x2 <= P_wind
        total = x1 + x2
        over  = total > self.d                         # viola x1+x2 <= d_t
        # Reescalado proporcional solo donde se incumple (evita division por 0)
        scale = np.where(over & (total > 0), self.d / np.where(total > 0, total, 1.0), 1.0)
        x1 = x1 * scale
        x2 = x2 * scale
        return x1, x2

    # ----------------------------------------------------------------
    def evaluate(self, solution: FloatSolution) -> FloatSolution:
        x1, x2 = self._repair(solution.variables)
        # Actualizar el cromosoma para que la descendencia herede genes válidos
        solution.variables = np.concatenate([x1, x2]).tolist()

        grid = self.d - x1 - x2                        # energía de red (>= 0)

        f1 = float(self.a1 @ x1 + self.a2 @ x2 + self.a3 @ grid)  # coste total (€)
        f2 = float(grid.sum())                                    # energía de red (kWh)

        solution.objectives[0] = f1
        solution.objectives[1] = f2
        return solution


# Instanciar el problema sobre la ventana cargada
problem = MicrogridDispatch(DEMAND, P_SOLAR, P_WIND, A_SOLAR, A_WIND, A_GRID)
print(f'Problema listo: {problem.name()}')
print(f'Variables: {problem.number_of_variables()}  |  Objetivos: {problem.number_of_objectives()}')

## 4. Cotas del espacio objetivo y punto de referencia del HV

In [ ]:
# ============================================================
#  COTAS DEL ESPACIO OBJETIVO Y PUNTO DE REFERENCIA DEL HV
# ============================================================
# El problema es SEPARABLE hora a hora, así que los extremos de cada objetivo
# se calculan analíticamente (exactos, deterministas):
#
#   f2 (energía de red):
#     min = sum_t max(0, d_t - P_solar_t - P_wind_t)   (renovables al máximo)
#     max = sum_t d_t                                  (todo de la red)
#   f1 (coste): por hora se rellena la demanda
#     min -> fuentes mas baratas que la red primero
#     max -> fuentes mas caras que la red primero  (peor rincón factible)
#
# El punto de referencia del HV es el peor rincón (f1_max, f2_max) + margen,
# que GARANTIZA dominar todo el frente (mejor que muestrear: el muestreo aleatorio
# tiende a sobreusar renovables e infraestima el extremo de mucha energía de red).

def objective_bounds(d, ps, pw, a_s, a_w, a_g):
    f1_min = f1_max = 0.0
    for t in range(len(d)):
        srcs = [(a_s[t], ps[t]), (a_w[t], pw[t])]
        rem, c = d[t], 0.0                                   # coste mínimo
        for price, cap in sorted(srcs, key=lambda z: z[0]):
            if price < a_g[t]:
                u = min(cap, rem); c += price * u; rem -= u
        c += a_g[t] * rem; f1_min += c
        rem, c = d[t], 0.0                                   # coste máximo
        for price, cap in sorted(srcs, key=lambda z: -z[0]):
            if price > a_g[t]:
                u = min(cap, rem); c += price * u; rem -= u
        c += a_g[t] * rem; f1_max += c
    f2_min = float(np.maximum(0.0, d - ps - pw).sum())
    f2_max = float(d.sum())
    return f1_min, f1_max, f2_min, f2_max

MARGIN = 0.1
F1_MIN, F1_MAX, F2_MIN, F2_MAX = objective_bounds(DEMAND, P_SOLAR, P_WIND, A_SOLAR, A_WIND, A_GRID)
area_espacio    = (F1_MAX - F1_MIN) * (F2_MAX - F2_MIN)
REFERENCE_POINT = np.array([F1_MAX, F2_MAX]) * (1 + MARGIN)

print(f'f1 (coste)       : min {F1_MIN:.4f} €   | max {F1_MAX:.4f} €')
print(f'f2 (energia red) : min {F2_MIN:.2f} kWh | max {F2_MAX:.2f} kWh')
print(f'Área espacio objetivo: {area_espacio:.4f}')
print(f'Punto de referencia HV: coste={REFERENCE_POINT[0]:.4f} €, energia_red={REFERENCE_POINT[1]:.2f} kWh')

## 5. Función auxiliar: ejecutar un algoritmo y calcular su HV

In [ ]:
# ============================================================
#  EJECUTAR UN ALGORITMO Y DEVOLVER SU HYPERVOLUME
# ============================================================
def run_algorithm(algo_name: str, population_size: int, eta_c: float, eta_m: float,
                  crossover_prob: float, seed: int) -> dict:
    """Ejecuta NSGA-II o SPEA2 con los hiperparámetros dados y devuelve HV, nº de
    soluciones y tiempo. La población es un PARÁMETRO FIJO (no se busca)."""
    random.seed(seed)
    np.random.seed(seed)

    mutation_prob   = 1.0 / problem.number_of_variables()   # regla estándar 1/n
    max_evaluations = population_size * GENERATIONS          # mismo presupuesto para ambos

    crossover_op = SBXCrossover(probability=crossover_prob, distribution_index=eta_c)
    mutation_op  = PolynomialMutation(probability=mutation_prob, distribution_index=eta_m)
    termination  = StoppingByEvaluations(max_evaluations=max_evaluations)

    t0 = time.time()
    if algo_name == 'NSGAII':
        algo = NSGAII(problem=problem, population_size=population_size,
                      offspring_population_size=population_size,
                      mutation=mutation_op, crossover=crossover_op,
                      termination_criterion=termination)
    elif algo_name == 'SPEA2':
        algo = SPEA2(problem=problem, population_size=population_size,
                     offspring_population_size=population_size,
                     mutation=mutation_op, crossover=crossover_op,
                     termination_criterion=termination)
    else:
        raise ValueError(f'Algoritmo desconocido: {algo_name}')

    algo.run()
    elapsed = time.time() - t0

    front     = get_non_dominated_solutions(algo.result())
    front_arr = np.array([s.objectives for s in front])
    hv_ind    = HyperVolume(REFERENCE_POINT.tolist())
    hv        = hv_ind.compute(front_arr.tolist()) if len(front) > 0 else 0.0

    return {'hv': hv, 'n_solutions': len(front), 'elapsed_s': elapsed}

print('Función run_algorithm definida.')

## 6. Búsqueda con Optuna

In [ ]:
# ============================================================
#  BÚSQUEDA DE HIPERPARÁMETROS CON OPTUNA (TPE)
# ============================================================
# Un study por algoritmo (para conservar la comparación NSGA-II vs SPEA2).
# Se buscan eta_c, eta_m y crossover_prob (rangos CONTINUOS).
# La población NO se busca: queda fija en POPULATION_SIZE.
# Objetivo de Optuna: MAXIMIZAR la MEDIANA del HV sobre N_RUNS semillas.

optuna.logging.set_verbosity(optuna.logging.WARNING)

def make_objective(algo_name):
    def objective(trial):
        eta_c = trial.suggest_float('eta_c',          *SEARCH_SPACE['eta_c'])
        eta_m = trial.suggest_float('eta_m',          *SEARCH_SPACE['eta_m'])
        cx    = trial.suggest_float('crossover_prob', *SEARCH_SPACE['crossover_prob'])

        hv_runs, elapsed_runs = [], []
        for r in range(N_RUNS):
            seed = BASE_SEED + r * 1000
            try:
                out = run_algorithm(algo_name, POPULATION_SIZE, eta_c, eta_m, cx, seed)
                hv_runs.append(out['hv'])
                elapsed_runs.append(out['elapsed_s'])
            except Exception as e:
                print(f'  Error en {algo_name} run {r}: {e}')
                hv_runs.append(np.nan); elapsed_runs.append(np.nan)

        trial.set_user_attr('hv_mean',        float(np.nanmean(hv_runs)))
        trial.set_user_attr('hv_std',         float(np.nanstd(hv_runs)))
        trial.set_user_attr('hv_min',         float(np.nanmin(hv_runs)))
        trial.set_user_attr('hv_max',         float(np.nanmax(hv_runs)))
        trial.set_user_attr('elapsed_mean_s', float(np.nanmean(elapsed_runs)))
        trial.set_user_attr('hv_runs',        [float(h) for h in hv_runs])
        return float(np.nanmedian(hv_runs))   # se MAXIMIZA
    return objective

results   = []   # una fila por trial (de los dos algoritmos)
studies   = {}
best_params = {}

for algo_name in ALGORITHMS:
    print(f"\n{'='*60}\n Optuna -> {algo_name}\n{'='*60}")
    sampler = optuna.samplers.TPESampler(seed=BASE_SEED)
    study   = optuna.create_study(direction='maximize', sampler=sampler, study_name=algo_name)
    study.optimize(make_objective(algo_name), n_trials=N_TRIALS, show_progress_bar=True)

    studies[algo_name]     = study
    best_params[algo_name] = study.best_params
    print(f'  Mejor HV (mediana) {algo_name}: {study.best_value:.6f}')
    print(f'  Mejores parámetros : {study.best_params}')

    for t in study.trials:
        results.append({
            'algorithm'      : algo_name,
            'population_size': POPULATION_SIZE,
            'eta_c'          : t.params.get('eta_c'),
            'eta_m'          : t.params.get('eta_m'),
            'crossover_prob' : t.params.get('crossover_prob'),
            'hv_median'      : t.value,
            'hv_mean'        : t.user_attrs.get('hv_mean'),
            'hv_std'         : t.user_attrs.get('hv_std'),
            'hv_min'         : t.user_attrs.get('hv_min'),
            'hv_max'         : t.user_attrs.get('hv_max'),
            'elapsed_mean_s' : t.user_attrs.get('elapsed_mean_s'),
            'hv_runs'        : t.user_attrs.get('hv_runs'),
        })

print('\nBúsqueda Optuna completada.')

## 7. Guardar resultados

In [ ]:
# ============================================================
#  GUARDAR RESULTADOS
# ============================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.DataFrame(results).sort_values(
    ['algorithm', 'hv_median'], ascending=[True, False]).reset_index(drop=True)

ruta_csv = os.path.join(OUTPUT_DIR, 'resultados_optuna_microred.csv')
ruta_pkl = os.path.join(OUTPUT_DIR, 'resultados_optuna_microred.pkl')
df.to_csv(ruta_csv, index=False)
df.to_pickle(ruta_pkl)
print(f'Guardado:\n  {ruta_csv}\n  {ruta_pkl}')

# Guardar también la mejor configuración por algoritmo
with open(os.path.join(OUTPUT_DIR, 'best_params_microred.json'), 'w') as f:
    json.dump(best_params, f, indent=2)
print('  best_params_microred.json')
display(df.groupby('algorithm').head(5))